In [ ]:
# !pip install chromadb
# !pip install -U sentence-transformers transformers huggingface_hub


In [21]:
from pathlib import Path
import json

import chromadb
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [22]:

# --- Paths ---
project_root = Path("/home/milijanas/projects/genomic-rag-assistant")
data_raw = project_root / "data/raw"
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

In [23]:
UNIPROT_PARQUET = data_raw / "uniprot_human_reviewed.parquet"
UNIPROT_EXPORT = data_processed / "docs_for_rag_uniprot.jsonl"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_EMBEDDINGS = data_processed / "uniprot_chunk_embeddings.npy"

UNIPROT_STREAM_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?compressed=true"
    "&fields=accession,id,gene_names,protein_name,length,"
    "cc_function,cc_disease,cc_subcellular_location,cc_interaction,"
    "ft_domain,ft_region,ft_zn_fing,cc_domain,"
    "ft_variant,cc_polymorphism,"
    "cc_tissue_specificity,cc_developmental_stage"
    "&format=tsv"
    "&query=(organism_id:9606)%20AND%20(reviewed:true)"
)

REQUIRED_COLUMNS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]


## Offline — build index

Run this section rarely: download → parquet → filter → chunk → embed → Chroma.

Set `DOWNLOAD = True` to refresh from UniProt. Set `REBUILD_INDEX = True` to re-embed and rebuild Chroma.


In [25]:
# OFFLINE: fetch UniProt stream and save parquet (slow; ~20k reviewed human proteins)
DOWNLOAD = False

if DOWNLOAD:
    df_raw = pd.read_csv(UNIPROT_STREAM_URL, sep="\t", compression="gzip", low_memory=False)
    data_raw.mkdir(parents=True, exist_ok=True)
    df_raw.to_parquet(UNIPROT_PARQUET, index=False)
    print("Saved parquet:", UNIPROT_PARQUET, df_raw.shape)
else:
    print("Skipping download. Set DOWNLOAD = True to refresh:", UNIPROT_PARQUET)


Saved parquet: /home/milijanas/projects/genomic-rag-assistant/data/raw/uniprot_human_reviewed.parquet (20431, 17)


In [ ]:
# OFFLINE: load parquet and subset for indexing
df = pd.read_parquet(UNIPROT_PARQUET)

In [27]:
# Optional: 
# df.head() 
# df[df["Gene Names"].str.contains("BRCA", na=False)]
# df[df["Gene Names"].str.contains("ERVK-19", na=False)]
df[ (df['Entry'] == 'P08865') | (df['Entry'] == 'A0A8I5KQE6') ]


,Entry,Entry Name,Gene Names,Protein names,Length,Function [CC],Involvement in disease,Subcellular location [CC],Interacts with,Domain [FT],Region,Zinc finger,Domain [CC],Natural variant,Polymorphism,Tissue specificity,Developmental stage
8,A0A8I5KQE6,RPSA2_HUMAN,RPSA2 RPSA RPSAP58,Small ribosomal subunit protein uS2B (37 kDa l...,295,FUNCTION: Required for the assembly and/or sta...,None,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...,None,None,"REGION 54..113; /note=""Interaction with PPP1R1...",None,None,None,None,None,None
2358,P08865,RSSA_HUMAN,RPSA LAMBR LAMR1,Small ribosomal subunit protein uS2 (37 kDa la...,295,FUNCTION: Required for the assembly and/or sta...,"DISEASE: Asplenia, isolated congenital (ICAS) ...",SUBCELLULAR LOCATION: Cell membrane. Cytoplasm...,Q8IYE1; P02489; Q9UQC2; P04792; Q15046; Q9Y2W7...,None,"REGION 54..113; /note=""Interaction with PPP1R1...",None,None,"VARIANT 54; /note=""T -> N (in ICAS; reduced pr...",None,None,None


In [ ]:
# Proteins with disease annotation (smaller, faster index)
df = df[df["Involvement in disease"].notna()].copy()
print(df.shape)
df.columns.tolist()

In [ ]:
def _is_valid_field(val) -> bool:
    if pd.isna(val):
        return False
    s = str(val).strip().lower()
    return bool(s) and s != "nan"


def _col_value(row, column: str) -> str:
    if column not in row.index:
        return ""
    return str(row[column]) if _is_valid_field(row.get(column)) else ""


def _first_gene(gene_names) -> str:
    if not _is_valid_field(gene_names):
        return ""
    return str(gene_names).split()[0]


def _merged_chunk_text(row, columns: list[str]) -> str:
    parts = []
    for column in columns:
        value = _col_value(row, column)
        if value:
            parts.append(f"{column}:\n{value}")
    return "\n\n".join(parts)


def _build_header(row) -> tuple[str, str, str, int, str]:
    entry_name = _col_value(row, "Entry Name") or str(row["Entry"])
    accession = str(row["Entry"])
    gene = _first_gene(row["Gene Names"])
    protein = _col_value(row, "Protein names")
    length = int(row["Length"]) if pd.notna(row.get("Length")) else 0
    header = f"Entry name: {entry_name}\nAccession: {accession}"
    if gene:
        header += f"\nGene: {gene}"
    if protein:
        header += f"\nProtein: {protein}"
    if length:
        header += f"\nLength: {length}"
    return entry_name, accession, gene, length, header


def build_chunked_docs(dataframe: pd.DataFrame) -> list:
    chunked = []
    chunk_specs = [
        ("identity", ["Entry Name", "Gene Names", "Protein names", "Length"]),
        ("function", ["Function [CC]"]),
        ("disease", ["Involvement in disease"]),
        # ("structure", ["Domain [FT]", "Zinc finger"]),
        ("expression", ["Tissue specificity"]),
    ]
    for _, row in dataframe.iterrows():
        entry_name, accession, gene, length, header = _build_header(row)
        for chunk_type, columns in chunk_specs:
            if chunk_type == "identity":
                text = f"{header}\nType: identity"
            else:
                body = _merged_chunk_text(row, columns)
                if not body:
                    continue
                text = f"{header}\nType: {chunk_type}\n{body}"
            chunked.append({
                "id": f"{entry_name}_{chunk_type}",
                "text": text,
                "metadata": {
                    "entry_name": entry_name,
                    "accession": accession,
                    "chunk_type": chunk_type,
                    "gene": gene,
                    "length": length,
                },
            })
    return chunked


In [ ]:
# OFFLINE: chunk, export jsonl, embed, write Chroma
REBUILD_INDEX = True
CHROMA_BATCH_SIZE = 5000
EMBED_BATCH_SIZE = 64

docs = build_chunked_docs(df)
print(f"Built {len(docs)} chunks from {len(df)} proteins (~{len(docs) / len(df):.1f} chunks/protein)")

data_processed.mkdir(parents=True, exist_ok=True)
chroma_dir.mkdir(parents=True, exist_ok=True)

with UNIPROT_EXPORT.open("w", encoding="utf-8") as f:
    for d in docs:
        f.write(json.dumps({"id": d["id"], "text": d["text"], "metadata": d["metadata"]}, ensure_ascii=False) + "\n")
print("Wrote", UNIPROT_EXPORT)

ids = [d["id"] for d in docs]
documents = [d["text"] for d in docs]
metadatas = [d["metadata"] for d in docs]

client = chromadb.PersistentClient(path=str(chroma_dir))

if REBUILD_INDEX:
    model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    embeddings = model.encode(
        documents,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    np.save(UNIPROT_EMBEDDINGS, embeddings)
    print("Saved embeddings:", UNIPROT_EMBEDDINGS)

    try:
        client.delete_collection(UNIPROT_COLLECTION)
    except Exception:
        pass
    collection = client.create_collection(UNIPROT_COLLECTION)
    for i in range(0, len(ids), CHROMA_BATCH_SIZE):
        collection.add(
            ids=ids[i : i + CHROMA_BATCH_SIZE],
            documents=documents[i : i + CHROMA_BATCH_SIZE],
            embeddings=embeddings[i : i + CHROMA_BATCH_SIZE].tolist(),
            metadatas=metadatas[i : i + CHROMA_BATCH_SIZE],
        )
    print(f"Indexed {len(ids)} chunks -> {UNIPROT_COLLECTION}")
else:
    print("REBUILD_INDEX is False — skipping embed/Chroma. Run online section only.")


## Online — query only

Uses the existing Chroma collection. No full re-embed. Load full protein rows from parquet after hits.


In [ ]:
# ONLINE: semantic search (query embedding only)
query = "proteins related to breast cancer"
TOP_K_CHUNKS = 20
TOP_K_PROTEINS = 5

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path=str(chroma_dir))
collection = client.get_collection(UNIPROT_COLLECTION)
print(f"Collection {UNIPROT_COLLECTION}: {collection.count()} chunks")

results = collection.query(
    query_embeddings=[model.encode(query).tolist()],
    n_results=TOP_K_CHUNKS,
    include=["documents", "metadatas", "distances"],
)

best_by_entry_name = {}
for doc_text, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    entry_name = (meta or {}).get("entry_name")
    if entry_name not in best_by_entry_name or dist < best_by_entry_name[entry_name]["distance"]:
        best_by_entry_name[entry_name] = {"text": doc_text, "metadata": meta, "distance": dist}

ranked = sorted(best_by_entry_name.values(), key=lambda x: x["distance"])[:TOP_K_PROTEINS]
print(f"Query: {query}\n")
for rank, hit in enumerate(ranked, 1):
    m = hit["metadata"] or {}
    print(
        f"{rank}. entry_name={m.get('entry_name')} accession={m.get('accession')} "
        f"gene={m.get('gene')} chunk={m.get('chunk_type')} dist={hit['distance']:.4f}"
    )
    print(hit["text"][:500] + ("..." if len(hit["text"]) > 500 else ""))
    print()


In [ ]:
# ONLINE: hydrate full records from parquet for top hits
# Re-run previous cell first, or set accessions manually.
if "ranked" not in globals():
    raise ValueError("Run the query cell above first.")

full_df = pd.read_parquet(UNIPROT_PARQUET)
for rank, hit in enumerate(ranked, 1):
    acc = (hit["metadata"] or {}).get("accession")
    row = full_df.loc[full_df["Entry"] == acc]
    print(f"\n--- {rank}. {acc} ({len(row)} row(s)) ---")
    if len(row):
        print(row.iloc[0].to_string()[:2000])


add queries for each of the 4 chunk types, with interesting proteins that you know about already.
        ("identity", ["Entry Name", "Gene Names", "Protein names", "Length"]),
        ("function", ["Function [CC]"]),
        ("disease", ["Involvement in disease"]),
        ("expression", ["Tissue specificity"]),

retreive full uniprot entry for the selected chunk
use open source llm to summarize into natural language

convert into src + eda nb

document:
md summary (methodology, filters, examples)
comments
readme

suggested improvements - performance

combine both into 1 rag:
{
    "entity_type": "clinvar_variant",

    "title": variant_name,

    "text": generated_text,

    "gene": gene,

    "diseases": phenotype_list,

    "source": "clinvar",

    "metadata": {...}
}
 